In [1]:
from google.colab import drive
drive.mount('/content/drive')

#!pip install torch torchvision torchaudio
!pip install matplotlib numpy pillow scikit-image opencv-python pandas seaborn imageio
!pip install -q imageio-ffmpeg

Mounted at /content/drive


In [2]:
import sys
import os
sys.path.append('/content/drive/MyDrive/ResearchProject')

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import time

from unet_denoiser_v2 import BlindVideoDenoiserUNet
from video_io import VideoLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

project_dir = Path('/content/drive/MyDrive/ResearchProject')
benchmark_dir = project_dir / 'benchmark_results_inpainting'
benchmark_dir.mkdir(exist_ok=True)

Using device: cuda
GPU: NVIDIA A100-SXM4-40GB
GPU Memory: 42.4 GB


In [3]:
from inverse_problem_framework import LinearOperator, VideoDenoiser, KadkhodaieSolver
from linear_operators import InpaintingOperator, create_center_mask, create_inpainting_degradation
from skimage.metrics import peak_signal_noise_ratio as compute_psnr
from skimage.metrics import structural_similarity as compute_ssim

print("✓ Framework imported (KadkhodaieSolver for inpainting)")

✓ Framework imported (KadkhodaieSolver for inpainting)


In [4]:
class BlindVideoDenoiserWrapper(VideoDenoiser):
    def __init__(self, model, device='cuda', pad_to=32, num_input_frames=5):
        self.model = model.to(device)
        self.device = device
        self.pad_to = pad_to
        self.num_input_frames = num_input_frames
        self.half_window = num_input_frames // 2
        self.model.eval()

    def _pad(self, frame):
        C, H, W = frame.shape
        ph = (self.pad_to - H % self.pad_to) % self.pad_to
        pw = (self.pad_to - W % self.pad_to) % self.pad_to
        if ph > 0 or pw > 0:
            frame = F.pad(frame, (0, pw, 0, ph), mode='reflect')
        return frame, (H, W)

    def _unpad(self, frame, orig):
        return frame[:, :orig[0], :orig[1]]

    def denoise(self, noisy_video, noise_std=0.0):
        T, C, H, W = noisy_video.shape
        noisy_video = noisy_video.to(self.device)
        frames = []
        with torch.no_grad():
            for t in range(T):
                indices = [max(0, min(T-1, t+off)) for off in range(-self.half_window, self.half_window+1)]
                neighbor_frames = [noisy_video[i] for i in indices]
                padded = []
                orig_size = None
                for f in neighbor_frames:
                    pf, orig_size = self._pad(f)
                    padded.append(pf)
                concat = torch.cat(padded, dim=0).unsqueeze(0)
                out = self.model(concat)
                denoised = self._unpad(out.squeeze(0), orig_size)
                frames.append(torch.clamp(denoised, 0, 1))
        return torch.stack(frames)

    def denoise_frame(self, prev_frame, curr_frame, next_frame, noise_std=0.0):
        video = torch.stack([prev_frame, prev_frame, curr_frame, next_frame, next_frame])
        return self.denoise(video, noise_std)[2]

print("✓ Denoiser wrapper defined (5-frame, pad_to=32)")

✓ Denoiser wrapper defined (5-frame, pad_to=32)


In [5]:
print("Loading 5-frame blind video denoiser...")
checkpoint_path = project_dir / 'checkpoints_large_model' / 'best_model_final.pt'

model = BlindVideoDenoiserUNet(num_input_frames=5, out_channels=3, base_channels=64)
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

print(f"✓ Loaded from epoch {checkpoint['epoch']}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

your_denoiser = BlindVideoDenoiserWrapper(model, device=device, pad_to=32, num_input_frames=5)
print("✓ Wrapped (5 frames, pad_to=32)")

Loading 5-frame blind video denoiser...
✓ Loaded from epoch 26
  Parameters: 18,693,504
✓ Wrapped (5 frames, pad_to=32)


In [6]:
print("Loading test videos...")
davis_root = Path('/content/drive/MyDrive/ResearchProject/DAVISDataset')
test_videos = {}
video_dirs = sorted([d for d in davis_root.iterdir() if d.is_dir()])[:5]

for vdir in video_dirs:
    name = vdir.name
    print(f"  Loading {name}...", end=' ', flush=True)
    try:
        v = VideoLoader.load_frame_sequence(
            str(vdir), max_frames=None, resize=(512, 832), device=device
        )
        test_videos[name] = v
        print(f"✓ {v.shape}")
    except Exception as e:
        print(f"✗ ({e})")

print(f"\nLoaded {len(test_videos)} test videos")

Loading test videos...
  Loading baseball... Loaded 90 frames from: /content/drive/MyDrive/ResearchProject/DAVISDataset/baseball
  Shape: torch.Size([90, 3, 512, 832]) (T, C, H, W)
✓ torch.Size([90, 3, 512, 832])
  Loading basketball-game... Loaded 77 frames from: /content/drive/MyDrive/ResearchProject/DAVISDataset/basketball-game
  Shape: torch.Size([77, 3, 512, 832]) (T, C, H, W)
✓ torch.Size([77, 3, 512, 832])
  Loading bear... Loaded 82 frames from: /content/drive/MyDrive/ResearchProject/DAVISDataset/bear
  Shape: torch.Size([82, 3, 512, 832]) (T, C, H, W)
✓ torch.Size([82, 3, 512, 832])
  Loading bears-ball... Loaded 78 frames from: /content/drive/MyDrive/ResearchProject/DAVISDataset/bears-ball
  Shape: torch.Size([78, 3, 512, 832]) (T, C, H, W)
✓ torch.Size([78, 3, 512, 832])
  Loading bike-packing... Loaded 69 frames from: /content/drive/MyDrive/ResearchProject/DAVISDataset/bike-packing
  Shape: torch.Size([69, 3, 512, 832]) (T, C, H, W)
✓ torch.Size([69, 3, 512, 832])

Loaded 5

In [7]:
print("=" * 60)
print("INPAINTING TEST (Center hole 160×288)")
print("=" * 60)

hole_h, hole_w = 40, 70

all_inpaint_results = {}

for video_name, clean in test_videos.items():
    T, C, H, W = clean.shape
    print(f"\n--- {video_name} ({T} frames, {W}×{H}) ---")

    operator, masked_video, mask = create_inpainting_degradation(
        clean, H, W, hole_h=hole_h, hole_w=hole_w, device=device
    )
    observed_frac = mask.mean().item()
    print(f"  Hole: {hole_w}×{hole_h} center rectangle, {observed_frac:.1%} observed")

    solver = KadkhodaieSolver(operator, your_denoiser, device=device)

    print("  Solving...")
    restored, metrics = solver.solve(
        masked_video,
        sigma_0=0.15,
        sigma_L=0.0003,
        h0=0.01,
        beta=0.1,
        max_iterations=2000,
        verbose=True,
        log_freq=200,
    )

    all_inpaint_results[video_name] = {
        'clean': clean,
        'masked': masked_video,
        'mask': mask,
        'restored': restored,
        'metrics': metrics,
    }

    # Quick summary
    mid = T // 2
    c = clean[mid].permute(1, 2, 0).cpu().numpy()
    m = masked_video[mid].permute(1, 2, 0).cpu().numpy()
    r = restored[mid].permute(1, 2, 0).cpu().numpy()
    print(f"  Masked:  {compute_psnr(c, np.clip(m,0,1), data_range=1.0):.2f} dB")
    print(f"  Solver:  {compute_psnr(c, np.clip(r,0,1), data_range=1.0):.2f} dB")

print(f"\n{'=' * 60}")
print(f"Done — {len(all_inpaint_results)} videos processed")
print(f"{'=' * 60}")

INPAINTING TEST (Center hole 160×288)

--- baseball (90 frames, 832×512) ---
  Hole: 70×40 center rectangle, 99.3% observed
  Solving...
  Converged in 155 iters (244.7s), final sigma=0.000290
  Masked:  27.91 dB
  Solver:  43.00 dB

--- basketball-game (77 frames, 832×512) ---
  Hole: 70×40 center rectangle, 99.3% observed
  Solving...
  Converged in 155 iters (208.8s), final sigma=0.000288
  Masked:  30.33 dB
  Solver:  33.79 dB

--- bear (82 frames, 832×512) ---
  Hole: 70×40 center rectangle, 99.3% observed
  Solving...
  Converged in 154 iters (221.0s), final sigma=0.000300
  Masked:  25.91 dB
  Solver:  36.84 dB

--- bears-ball (78 frames, 832×512) ---
  Hole: 70×40 center rectangle, 99.3% observed
  Solving...
  Converged in 155 iters (211.6s), final sigma=0.000287
  Masked:  31.44 dB
  Solver:  33.26 dB

--- bike-packing (69 frames, 832×512) ---
  Hole: 70×40 center rectangle, 99.3% observed
  Solving...
  Converged in 155 iters (187.3s), final sigma=0.000287
  Masked:  30.34 d

In [8]:
frames_per_video = 6

for video_name, data in all_inpaint_results.items():
    clean = data['clean']
    masked = data['masked']
    restored = data['restored']
    T = clean.shape[0]

    step = max(1, T // frames_per_video)
    frame_indices = list(range(0, T, step))[:frames_per_video]

    print(f"\n--- {video_name} ({T} frames) ---")

    for idx in frame_indices:
        c_np = clean[idx].permute(1, 2, 0).cpu().numpy()
        m_np = masked[idx].permute(1, 2, 0).cpu().numpy()
        r_np = restored[idx].permute(1, 2, 0).cpu().numpy()

        p_m = compute_psnr(c_np, np.clip(m_np, 0, 1), data_range=1.0)
        p_r = compute_psnr(c_np, np.clip(r_np, 0, 1), data_range=1.0)

        fig, axes = plt.subplots(1, 3, figsize=(20, 7))

        axes[0].imshow(np.clip(c_np, 0, 1))
        axes[0].set_title('Clean (Ground Truth)', fontsize=13, fontweight='bold')
        axes[0].axis('off')

        axes[1].imshow(np.clip(m_np, 0, 1))
        axes[1].set_title(f'Masked — PSNR={p_m:.1f} dB', fontsize=13, fontweight='bold')
        axes[1].axis('off')

        axes[2].imshow(np.clip(r_np, 0, 1))
        axes[2].set_title(f'Solver Output — PSNR={p_r:.1f} dB', fontsize=13, fontweight='bold')
        axes[2].axis('off')

        plt.suptitle(f'{video_name} — Frame {idx+1}/{T} (Inpainting)',
                     fontsize=15, fontweight='bold')
        plt.tight_layout()
        plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [9]:
from PIL import Image

frames_per_video = 6
save_root = str(benchmark_dir / 'inpainting_frames_more_iterations')

for video_name, data in all_inpaint_results.items():
    clean = data['clean']
    masked = data['masked']
    restored = data['restored']
    T = clean.shape[0]

    step = max(1, T // frames_per_video)
    frame_indices = list(range(0, T, step))[:frames_per_video]

    video_dir = os.path.join(save_root, video_name)
    os.makedirs(video_dir, exist_ok=True)

    for idx in frame_indices:
        versions = {
            'clean': clean[idx],
            'masked': masked[idx],
            'solver_YourUNet': restored[idx],
        }
        for label, tensor in versions.items():
            img_np = tensor.permute(1, 2, 0).cpu().numpy()
            img_uint8 = (np.clip(img_np, 0, 1) * 255).astype(np.uint8)
            Image.fromarray(img_uint8).save(
                os.path.join(video_dir, f'frame{idx+1:03d}_{label}.png')
            )

    print(f"Saved {len(frame_indices)} frames × {len(versions)} versions for {video_name}")

print(f"\nAll frames saved to: {save_root}")

Saved 6 frames × 3 versions for baseball
Saved 6 frames × 3 versions for basketball-game
Saved 6 frames × 3 versions for bear
Saved 6 frames × 3 versions for bears-ball
Saved 6 frames × 3 versions for bike-packing

All frames saved to: /content/drive/MyDrive/ResearchProject/benchmark_results_inpainting/inpainting_frames_more_iterations


In [10]:
import subprocess

output_dir = '/content/drive/MyDrive/ResearchProject/inpainting_videos_more_iterations'
os.makedirs(output_dir, exist_ok=True)
fps = 24

for video_name, data in all_inpaint_results.items():
    T = data['clean'].shape[0]

    videos_to_save = {
        'clean': data['clean'],
        'masked': data['masked'],
        'solver_YourUNet': data['restored'],
    }

    for label, tensor in videos_to_save.items():
        frames_np = [tensor[t].permute(1, 2, 0).cpu().numpy() for t in range(T)]
        h, w = frames_np[0].shape[:2]
        save_path = os.path.join(output_dir, f"{video_name}_inpaint_{label}.mp4")

        cmd = [
            'ffmpeg', '-y', '-f', 'rawvideo', '-vcodec', 'rawvideo',
            '-pix_fmt', 'rgb24', '-s', f'{w}x{h}', '-r', str(fps),
            '-i', '-', '-c:v', 'libx264', '-crf', '0',
            '-preset', 'medium', '-pix_fmt', 'yuv444p', save_path
        ]
        proc = subprocess.Popen(cmd, stdin=subprocess.PIPE,
                                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        for frame in frames_np:
            proc.stdin.write((np.clip(frame, 0, 1) * 255).astype(np.uint8).tobytes())
        proc.stdin.close()
        proc.wait()

    print(f"Saved {len(videos_to_save)} videos for {video_name}")

print(f"\nAll videos saved to: {output_dir}")

Saved 3 videos for baseball
Saved 3 videos for basketball-game
Saved 3 videos for bear
Saved 3 videos for bears-ball
Saved 3 videos for bike-packing

All videos saved to: /content/drive/MyDrive/ResearchProject/inpainting_videos_more_iterations


In [11]:
print("=" * 60)
print(f"INPAINTING — SUMMARY (hole {hole_w}×{hole_h})")
print("=" * 60)

rows = []
for video_name, data in all_inpaint_results.items():
    clean = data['clean']
    masked = data['masked']
    restored = data['restored']
    T = clean.shape[0]

    psnr_m, psnr_sol, ssim_m, ssim_sol = [], [], [], []
    for t in range(T):
        c = clean[t].permute(1, 2, 0).cpu().numpy()
        m = masked[t].permute(1, 2, 0).cpu().numpy()
        r = restored[t].permute(1, 2, 0).cpu().numpy()
        psnr_m.append(compute_psnr(c, np.clip(m, 0, 1), data_range=1.0))
        psnr_sol.append(compute_psnr(c, np.clip(r, 0, 1), data_range=1.0))
        ssim_m.append(compute_ssim(c, np.clip(m, 0, 1), data_range=1.0, channel_axis=2))
        ssim_sol.append(compute_ssim(c, np.clip(r, 0, 1), data_range=1.0, channel_axis=2))

    rows.append({
        'Video': video_name,
        'Frames': T,
        'PSNR Masked': np.mean(psnr_m),
        'PSNR Solver': np.mean(psnr_sol),
        'PSNR Gain': np.mean(psnr_sol) - np.mean(psnr_m),
        'SSIM Masked': np.mean(ssim_m),
        'SSIM Solver': np.mean(ssim_sol),
    })

    print(f"  {video_name}: Masked {np.mean(psnr_m):.2f} → Solver {np.mean(psnr_sol):.2f} dB "
          f"(Δ = {np.mean(psnr_sol)-np.mean(psnr_m):+.2f})")

df = pd.DataFrame(rows)
print(f"\nOverall average:")
print(f"  Masked: {df['PSNR Masked'].mean():.2f} dB | Solver: {df['PSNR Solver'].mean():.2f} dB | "
      f"Gain: {df['PSNR Gain'].mean():+.2f} dB")

df.to_csv(benchmark_dir / 'inpainting_benchmark.csv', index=False)
print(f"\nSaved to {benchmark_dir / 'inpainting_benchmark.csv'}")

INPAINTING — SUMMARY (hole 70×40)
  baseball: Masked 27.21 → Solver 40.85 dB (Δ = +13.64)
  basketball-game: Masked 29.88 → Solver 35.18 dB (Δ = +5.29)
  bear: Masked 27.00 → Solver 37.12 dB (Δ = +10.12)
  bears-ball: Masked 32.25 → Solver 33.79 dB (Δ = +1.54)
  bike-packing: Masked 30.56 → Solver 33.21 dB (Δ = +2.66)

Overall average:
  Masked: 29.38 dB | Solver: 36.03 dB | Gain: +6.65 dB

Saved to /content/drive/MyDrive/ResearchProject/benchmark_results_inpainting/inpainting_benchmark.csv


In [12]:
import openpyxl
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from datetime import datetime

wb = openpyxl.Workbook()
header_font = Font(bold=True, size=12)
title_font = Font(bold=True, size=14)
header_fill = PatternFill(start_color='D5E8F0', end_color='D5E8F0', fill_type='solid')
best_fill = PatternFill(start_color='C6EFCE', end_color='C6EFCE', fill_type='solid')
thin_border = Border(
    left=Side(style='thin'), right=Side(style='thin'),
    top=Side(style='thin'), bottom=Side(style='thin')
)

def style_cell(ws, row, col, fmt=None, bold=False, fill=None):
    cell = ws.cell(row=row, column=col)
    cell.border = thin_border
    cell.alignment = Alignment(horizontal='center')
    if fmt: cell.number_format = fmt
    if bold: cell.font = header_font
    if fill: cell.fill = fill

# ===== Sheet 1: PSNR =====
ws1 = wb.active
ws1.title = 'PSNR Comparison'
ws1.cell(row=1, column=1, value=f'Inpainting — PSNR Comparison (dB)').font = title_font

headers = ['Video', 'Frames', 'Masked', 'YourUNet Solver', 'PSNR Gain']
for col, h in enumerate(headers, 1):
    ws1.cell(row=3, column=col, value=h)
    ws1.cell(row=3, column=col).font = header_font
    ws1.cell(row=3, column=col).fill = header_fill
    ws1.cell(row=3, column=col).border = thin_border

for i, r in enumerate(rows):
    row = 4 + i
    ws1.cell(row=row, column=1, value=r['Video'])
    ws1.cell(row=row, column=2, value=r['Frames'])
    ws1.cell(row=row, column=3, value=round(r['PSNR Masked'], 2))
    ws1.cell(row=row, column=4, value=round(r['PSNR Solver'], 2))
    ws1.cell(row=row, column=5, value=round(r['PSNR Gain'], 2))
    gain_fill = best_fill if r['PSNR Gain'] > 0 else None
    for col in range(1, 6):
        style_cell(ws1, row, col, '0.00' if col >= 3 else None)
    style_cell(ws1, row, 4, '0.00', fill=gain_fill)
    style_cell(ws1, row, 5, '0.00', fill=gain_fill)

ar = 4 + len(rows)
ws1.cell(row=ar, column=1, value='AVERAGE')
ws1.cell(row=ar, column=3, value=round(df['PSNR Masked'].mean(), 2))
ws1.cell(row=ar, column=4, value=round(df['PSNR Solver'].mean(), 2))
ws1.cell(row=ar, column=5, value=round(df['PSNR Gain'].mean(), 2))
for col in range(1, 6):
    style_cell(ws1, ar, col, '0.00' if col >= 3 else None, bold=True)

ws1.column_dimensions['A'].width = 22
for c in 'BCDE':
    ws1.column_dimensions[c].width = 18

# ===== Sheet 2: SSIM =====
ws2 = wb.create_sheet('SSIM Comparison')
ws2.cell(row=1, column=1, value=f'Inpainting — SSIM Comparison').font = title_font

headers2 = ['Video', 'Frames', 'Masked', 'YourUNet Solver', 'SSIM Gain']
for col, h in enumerate(headers2, 1):
    ws2.cell(row=3, column=col, value=h)
    ws2.cell(row=3, column=col).font = header_font
    ws2.cell(row=3, column=col).fill = header_fill
    ws2.cell(row=3, column=col).border = thin_border

for i, r in enumerate(rows):
    row = 4 + i
    ssim_gain = r['SSIM Solver'] - r['SSIM Masked']
    ws2.cell(row=row, column=1, value=r['Video'])
    ws2.cell(row=row, column=2, value=r['Frames'])
    ws2.cell(row=row, column=3, value=round(r['SSIM Masked'], 4))
    ws2.cell(row=row, column=4, value=round(r['SSIM Solver'], 4))
    ws2.cell(row=row, column=5, value=round(ssim_gain, 4))
    gain_fill = best_fill if ssim_gain > 0 else None
    for col in range(1, 6):
        style_cell(ws2, row, col, '0.0000' if col >= 3 else None)
    style_cell(ws2, row, 4, '0.0000', fill=gain_fill)
    style_cell(ws2, row, 5, '0.0000', fill=gain_fill)

ar2 = 4 + len(rows)
ws2.cell(row=ar2, column=1, value='AVERAGE')
ws2.cell(row=ar2, column=3, value=round(df['SSIM Masked'].mean(), 4))
ws2.cell(row=ar2, column=4, value=round(df['SSIM Solver'].mean(), 4))
ws2.cell(row=ar2, column=5, value=round((df['SSIM Solver'] - df['SSIM Masked']).mean(), 4))
for col in range(1, 6):
    style_cell(ws2, ar2, col, '0.0000' if col >= 3 else None, bold=True)

ws2.column_dimensions['A'].width = 22
for c in 'BCDE':
    ws2.column_dimensions[c].width = 18

# ===== Sheet 3: Notes =====
ws3 = wb.create_sheet('Notes')
ws3.cell(row=1, column=1, value='Inpainting Benchmark — Methodology Notes').font = title_font
notes = [
    ('Date:', str(datetime.now())),
    ('Test videos:', str(len(rows))),
    ('Processing resolution:', f'{clean.shape[3]}×{clean.shape[2]}'),
    ('Hole size:', f'{hole_w}×{hole_h} (center rectangle)'),
    ('Observed fraction:', f'{(1 - (hole_h*hole_w)/(H*W)):.1%}'),
    ('Solver:', 'Kadkhodaie & Simoncelli (2021)'),
    ('sigma_0:', '0.15'), ('sigma_L:', '0.003'), ('h0:', '0.01'), ('beta:', '0.1'),
    ('', ''),
    ('Metrics:', ''),
    ('PSNR', 'Peak Signal-to-Noise Ratio vs clean ground truth (dB). Higher = better.'),
    ('SSIM', 'Structural Similarity vs clean ground truth (0 to 1). Higher = better.'),
    ('PSNR/SSIM Gain', 'Improvement of solver output over the masked input.'),
    ('', ''),
    ('Method Notes:', ''),
    ('Masked', 'Input observation: clean video with center rectangle zeroed out (black).'),
    ('YourUNet Solver', 'Kadkhodaie solver using denoiser prior to fill the missing region.'),
    ('', ''),
    ('Why Kadkhodaie works here:', ''),
    ('', 'Inpainting mask is a diagonal binary operator — exact projection.'),
    ('', 'Observed pixels are kept, missing pixels are filled by denoiser prior.'),
    ('', ''),
    ('Green cells', 'indicate positive improvement over the masked input.'),
]
for i, (k, val) in enumerate(notes):
    ws3.cell(row=3+i, column=1, value=k).font = Font(bold=True) if k else Font()
    ws3.cell(row=3+i, column=2, value=val)
ws3.column_dimensions['A'].width = 28
ws3.column_dimensions['B'].width = 75

excel_path = str(benchmark_dir / 'inpainting_benchmark.xlsx')
wb.save(excel_path)
print(f"✓ Excel saved: {excel_path}")
print(f"  Sheets: {wb.sheetnames}")

✓ Excel saved: /content/drive/MyDrive/ResearchProject/benchmark_results_inpainting/inpainting_benchmark.xlsx
  Sheets: ['PSNR Comparison', 'SSIM Comparison', 'Notes']
